# Nowcast + Scenario Workflow

This notebook demonstrates a **nowcasting-to-scenario pipeline**: predict the present,
then condition the future on alternative scenarios.

The workflow bridges two temporal domains:
1. **Nowcasting** — estimate the current quarter using high-frequency indicators
2. **Forecasting + Scenarios** — project forward from the nowcast under policy assumptions

**Pipeline**: Nowcast -> News Decomposition -> Bridge to Forecast -> Scenario Analysis -> Probability Assessment -> Monitoring

**Datasets used**: `mixed_freq.csv` (Phase 6), `us_macro_quarterly.csv` (Phase 5)

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# forecastbox nowcasting
from forecastbox.nowcasting import DFMNowcaster, NewsDecomposition

# forecastbox scenarios
from forecastbox.scenarios import (
    SimpleVAR,
    ConditionalForecast,
    ScenarioBuilder,
    MonteCarlo,
    FanChart,
)

# forecastbox metrics
from forecastbox.metrics import mae, rmse

# Helpers
sys.path.insert(0, "..")
from utils.helpers import load_all_datasets

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("All modules loaded successfully.")

## 1. The Nowcasting-Forecasting Bridge

In macroeconomic practice, the **present** is unknown: GDP is released with a 1-3 month
delay. Nowcasting fills this gap using timely monthly indicators.

Once we have a nowcast of the current quarter, we can **bridge** to a longer-horizon
forecast and attach **scenarios** for policy analysis:

```
Monthly data  -->  DFM Nowcast  -->  Bridge to VAR  -->  Scenarios  -->  Probabilities
(ragged edge)     (current GDP)    (forecast GDP)    (hawkish/dovish)   (P(recession))
```

This is exactly the pipeline central banks use for real-time assessment.

In [ ]:
# Load datasets
datasets = load_all_datasets()
mixed_freq = datasets["mixed_freq"]
us_macro = datasets["us_macro_quarterly"]

print("=== Mixed-Frequency Dataset ===")
print(f"Shape: {mixed_freq.shape}")
print(f"Date range: {mixed_freq.index[0]} to {mixed_freq.index[-1]}")
print(f"Columns: {list(mixed_freq.columns)}")
print(f"\nMissing values per column:")
print(mixed_freq.isna().sum())

print("\n=== US Macro Quarterly ===")
print(f"Shape: {us_macro.shape}")
print(f"Columns: {list(us_macro.columns)}")

# Visualize the ragged edge
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, mixed_freq.columns):
    series = mixed_freq[col].dropna()
    ax.plot(series.index, series.values, "steelblue", linewidth=1.2)
    ax.set_title(col.replace("_", " ").title(), fontsize=11)
    ax.grid(True, alpha=0.3)
    # Mark missing periods
    nan_mask = mixed_freq[col].isna()
    if nan_mask.any():
        for idx in mixed_freq.index[nan_mask]:
            ax.axvline(idx, color="red", alpha=0.1, linewidth=0.5)
fig.suptitle("Mixed-Frequency Data with Ragged Edge", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nRagged Edge (last 6 rows):")
print(mixed_freq.tail(6).to_string())

## 2. Step 1: Nowcast Current Quarter

We use a **Dynamic Factor Model (DFM)** to nowcast current-quarter GDP growth.
The DFM extracts latent factors from the mixed-frequency panel and handles the
ragged edge via the Kalman filter.

The Mariano-Murasawa (2003) accumulator links the quarterly GDP to the monthly factor.

In [ ]:
# Define frequency map
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

# Fit DFM
dfm = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)
dfm.fit(mixed_freq)
print(dfm)

# Nowcast
nowcast = dfm.nowcast(target="gdp_growth")
print(f"\n=== GDP Growth Nowcast ===")
print(f"Point estimate: {nowcast.point[0]:.4f}")
print(f"80% CI: [{nowcast.lower_80[0]:.4f}, {nowcast.upper_80[0]:.4f}]")
print(f"95% CI: [{nowcast.lower_95[0]:.4f}, {nowcast.upper_95[0]:.4f}]")

# Factor loadings
print("\nFactor Loadings:")
print(dfm.loadings())

# Plot factor vs GDP
factors = dfm.factors()
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(factors.index, factors["factor_1"], "steelblue", linewidth=2, label="Latent Factor")
ax1.set_ylabel("Factor", color="steelblue")
ax2 = ax1.twinx()
gdp_obs = mixed_freq["gdp_growth"].dropna()
ax2.scatter(gdp_obs.index, gdp_obs.values, color="darkorange", s=40, zorder=5, label="GDP (quarterly)")
# Mark nowcast
ax2.scatter([mixed_freq.index[-1]], [nowcast.point[0]], color="red", s=100, zorder=6,
            marker="*", label=f"Nowcast: {nowcast.point[0]:.2f}")
ax2.set_ylabel("GDP Growth", color="darkorange")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.set_title("DFM: Latent Factor vs GDP Growth (with Nowcast)", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Step 2: News Decomposition

When new data arrives, the nowcast revises. The **news decomposition** (Banbura & Modugno 2014)
tells us exactly **which data releases** drove the revision and by how much.

$$\Delta \hat{y} = \sum_i w_i \cdot (x_i^{\text{new}} - E[x_i | \Omega_{\text{old}}])$$

This is critical for central bank communication: "GDP was revised up because industrial
production surprised positively."

In [ ]:
# Simulate old vs new information sets
# Old: more missing data (before latest releases)
old_data = mixed_freq.copy()
old_data.iloc[-3:, old_data.columns.get_loc("industrial_production")] = np.nan
old_data.iloc[-3:, old_data.columns.get_loc("retail_sales")] = np.nan
old_data.iloc[-2:, old_data.columns.get_loc("confidence_index")] = np.nan

# New: some indicators updated (but not GDP)
new_data = mixed_freq.copy()
new_data.iloc[-1:, new_data.columns.get_loc("industrial_production")] = np.nan
new_data.iloc[-2:, new_data.columns.get_loc("retail_sales")] = np.nan

# Perform news decomposition
news_decomp = NewsDecomposition(dfm)
news_result = news_decomp.decompose(old_data, new_data, target="gdp_growth")

print("=== News Decomposition ===")
print(news_result.summary())

# Visualize contributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

news_result.plot_contributions(ax=axes[0])
axes[0].set_title("Contributions to Nowcast Revision", fontsize=12, fontweight="bold")

news_result.plot_waterfall(ax=axes[1])
axes[1].set_title("Waterfall: Old Nowcast -> New Nowcast", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nOld nowcast: {news_result.old_nowcast:.4f}")
print(f"New nowcast: {news_result.new_nowcast:.4f}")
print(f"Revision:    {news_result.total_revision:+.4f}")

## 4. Step 3: Bridge to Forecast

The nowcast gives us the **current quarter**. To forecast the future, we use a **VAR model**
on quarterly data, conditioning the first period on the nowcast.

This bridges the DFM's mixed-frequency nowcast into the VAR's multivariate forecast.

In [ ]:
# Build VAR on quarterly US macro data
var_vars = ["gdp_growth", "inflation", "fed_funds", "unemployment"]
endog = us_macro[var_vars].dropna().values
var_model = SimpleVAR(endog, p_order=2, var_names=var_vars)

forecast_steps = 8  # 8 quarters ahead

# Unconditional forecast (no bridge)
cf = ConditionalForecast(var_model, method="analytic")
unc_forecast = cf.forecast(steps=forecast_steps, conditions=None, n_draws=1000, seed=42)

# Bridged forecast: condition Q1 GDP on the nowcast
nowcast_value = float(nowcast.point[0])
bridged_forecast = cf.forecast(
    steps=forecast_steps,
    conditions={"gdp_growth": [nowcast_value]},  # condition only Q1
    n_draws=1000,
    seed=42,
)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
horizons = np.arange(1, forecast_steps + 1)

# GDP comparison
ax = axes[0]
ax.plot(horizons, unc_forecast["gdp_growth"].point, "b-o", label="Unconditional", linewidth=2)
ax.plot(horizons, bridged_forecast["gdp_growth"].point, "r-s", label=f"Bridged (nowcast={nowcast_value:.2f})", linewidth=2)
if bridged_forecast["gdp_growth"].lower_80 is not None:
    ax.fill_between(horizons, bridged_forecast["gdp_growth"].lower_80,
                    bridged_forecast["gdp_growth"].upper_80, alpha=0.15, color="red")
ax.axhline(nowcast_value, color="red", linestyle="--", alpha=0.5, label="Nowcast")
ax.set_title("GDP Growth: Unconditional vs Bridged", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)

# Inflation comparison
ax = axes[1]
ax.plot(horizons, unc_forecast["inflation"].point, "b-o", label="Unconditional", linewidth=2)
ax.plot(horizons, bridged_forecast["inflation"].point, "r-s", label="Bridged", linewidth=2)
ax.set_title("Inflation: Impact of GDP Bridge", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Inflation (%)")
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("Nowcast Bridge: Conditioning VAR on DFM Nowcast", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"GDP Q1 — Unconditional: {unc_forecast['gdp_growth'].point[0]:.4f}")
print(f"GDP Q1 — Bridged (nowcast): {bridged_forecast['gdp_growth'].point[0]:.4f}")

## 5. Step 4: Scenario Analysis

Starting from the nowcast, we build **three scenarios** for the future:

- **Baseline**: unconditional VAR forecast from the nowcast
- **Recession**: GDP contracts, unemployment rises, rates cut aggressively
- **Boom**: GDP accelerates, low unemployment, rates rise moderately

Each scenario conditions specific variables over the forecast horizon.

In [ ]:
# Build scenarios conditional on the nowcast
builder = ScenarioBuilder(var_model)

# Baseline: unconditional from nowcast bridge
baseline_gdp = bridged_forecast["gdp_growth"].point.tolist()
builder.add_scenario("baseline",
                     {"gdp_growth": [nowcast_value]},
                     description="Unconditional from nowcast")

# Recession: GDP negative, rising unemployment
recession_gdp = [nowcast_value - 0.5 * (t + 1) for t in range(forecast_steps)]
last_unemp = us_macro["unemployment"].iloc[-1]
recession_unemp = [last_unemp + 0.3 * (t + 1) for t in range(forecast_steps)]
builder.add_scenario("recession",
                     {"gdp_growth": recession_gdp, "unemployment": recession_unemp},
                     description="Contraction with rising unemployment")

# Boom: strong growth, low unemployment
boom_gdp = [nowcast_value + 0.4 * (t + 1) for t in range(forecast_steps)]
last_ff = us_macro["fed_funds"].iloc[-1]
boom_rates = [last_ff + 0.25 * (t + 1) for t in range(forecast_steps)]
builder.add_scenario("boom",
                     {"gdp_growth": boom_gdp, "fed_funds": boom_rates},
                     description="Strong expansion with rate hikes")

# Run all scenarios
scenario_results = builder.run(steps=forecast_steps, n_draws=1000, seed=42)

# Plot with fan charts
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
colors = {"baseline": "blue", "recession": "red", "boom": "green"}
horizons = np.arange(1, forecast_steps + 1)

for idx, var in enumerate(var_vars):
    ax = axes[idx // 2, idx % 2]
    for scen_name in ["baseline", "recession", "boom"]:
        fc = scenario_results.get(scen_name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
                label=scen_name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[scen_name])
    ax.set_title(var.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Scenario Analysis: Baseline vs Recession vs Boom (conditional on nowcast)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Step 5: Probability Assessment

Using **Monte Carlo simulation**, we compute the probability of adverse events
conditional on the current nowcast.

Key question: **What is P(recession) given our nowcast?**

We simulate 5,000 stochastic paths from the VAR and count the fraction that satisfy
each event definition at each horizon.

In [ ]:
# Monte Carlo simulation
mc = MonteCarlo(var_model, n_paths=5000, seed=42, parametric=True)
paths = mc.simulate(steps=forecast_steps)
print(f"Monte Carlo paths: {paths.shape}")
print(f"  -> {paths.shape[0]} paths x {paths.shape[1]} steps x {paths.shape[2]} variables")

# Event probabilities
prob_recession = mc.probability(lambda y: y < 0.0, variable="gdp_growth")
prob_high_inf = mc.probability(lambda y: y > 4.0, variable="inflation")
prob_high_unemp = mc.probability(lambda y: y > 8.0, variable="unemployment")

# Joint probability: recession AND high inflation (stagflation)
gdp_idx = var_vars.index("gdp_growth")
inf_idx = var_vars.index("inflation")
joint_mask = (paths[:, :, gdp_idx] < 0.0) & (paths[:, :, inf_idx] > 4.0)
prob_stagflation = joint_mask.mean(axis=0)

# Results table
prob_df = pd.DataFrame({
    "P(GDP < 0%)": prob_recession,
    "P(Inflation > 4%)": prob_high_inf,
    "P(Unemployment > 8%)": prob_high_unemp,
    "P(Stagflation)": prob_stagflation,
}, index=[f"Q+{h+1}" for h in range(forecast_steps)])

print("=== Conditional Event Probabilities (given nowcast) ===")
print(prob_df.round(4).to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Event probabilities by horizon
ax = axes[0]
ax.plot(horizons, prob_recession, "r-o", label="P(Recession)", linewidth=2)
ax.plot(horizons, prob_high_inf, "orange", marker="s", label="P(High Inflation)", linewidth=2)
ax.plot(horizons, prob_high_unemp, "purple", marker="^", label="P(High Unemployment)", linewidth=2)
ax.plot(horizons, prob_stagflation, "k--", marker="D", label="P(Stagflation)", linewidth=2)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Probability")
ax.set_title("Event Probabilities by Forecast Horizon", fontsize=12, fontweight="bold")
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: GDP density at Q+4 with recession threshold
ax = axes[1]
gdp_draws_q4 = paths[:, 3, gdp_idx]
ax.hist(gdp_draws_q4, bins=50, density=True, alpha=0.5, color="steelblue", edgecolor="white")
kde = gaussian_kde(gdp_draws_q4)
x_range = np.linspace(gdp_draws_q4.min() - 1, gdp_draws_q4.max() + 1, 200)
ax.plot(x_range, kde(x_range), "navy", linewidth=2)
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Recession threshold")
ax.fill_between(x_range[x_range < 0], kde(x_range[x_range < 0]), alpha=0.3, color="red",
                label=f"P(recession at Q+4) = {prob_recession[3]:.3f}")
ax.set_title("GDP Growth Density at Q+4", fontsize=12, fontweight="bold")
ax.set_xlabel("GDP Growth (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle("Probability Assessment via Monte Carlo (5,000 paths)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Step 6: Monitoring and Update

When a new data release arrives, the nowcast should be **updated** in real time.
We simulate the arrival of new industrial production data and show how the
nowcast, news decomposition, and scenario probabilities change.

This demonstrates the **monitoring loop** that central banks run continuously.

In [ ]:
# Simulate arrival of new data: industrial production for the latest month
# Create "before" and "after" datasets
data_before = mixed_freq.copy()
data_before.iloc[-2:, data_before.columns.get_loc("industrial_production")] = np.nan
data_before.iloc[-3:, data_before.columns.get_loc("retail_sales")] = np.nan

data_after = mixed_freq.copy()
data_after.iloc[-1:, data_after.columns.get_loc("industrial_production")] = np.nan
data_after.iloc[-2:, data_after.columns.get_loc("retail_sales")] = np.nan

# Nowcast BEFORE new data
dfm_before = DFMNowcaster(
    n_factors=1, factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum", em_iterations=100,
)
dfm_before.fit(data_before)
nowcast_before = dfm_before.nowcast(target="gdp_growth")

# Nowcast AFTER new data
dfm_after = DFMNowcaster(
    n_factors=1, factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum", em_iterations=100,
)
dfm_after.fit(data_after)
nowcast_after = dfm_after.nowcast(target="gdp_growth")

# News decomposition for the update
news_update = NewsDecomposition(dfm_after)
update_result = news_update.decompose(data_before, data_after, target="gdp_growth")

print("=== Nowcast Update ===")
print(f"Before new data: {nowcast_before.point[0]:.4f}  "
      f"(95% CI: [{nowcast_before.lower_95[0]:.4f}, {nowcast_before.upper_95[0]:.4f}])")
print(f"After new data:  {nowcast_after.point[0]:.4f}  "
      f"(95% CI: [{nowcast_after.lower_95[0]:.4f}, {nowcast_after.upper_95[0]:.4f}])")
print(f"Revision:        {update_result.total_revision:+.4f}")

# Update scenario probabilities with new nowcast
new_nowcast_value = float(nowcast_after.point[0])
mc_updated = MonteCarlo(var_model, n_paths=5000, seed=42, parametric=True)
paths_updated = mc_updated.simulate(steps=forecast_steps)
prob_recession_updated = mc_updated.probability(lambda y: y < 0.0, variable="gdp_growth")

# Compare before/after
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Nowcast comparison
ax = axes[0]
labels = ["Before", "After"]
points = [nowcast_before.point[0], nowcast_after.point[0]]
ci_low = [nowcast_before.lower_95[0], nowcast_after.lower_95[0]]
ci_high = [nowcast_before.upper_95[0], nowcast_after.upper_95[0]]
errors = [[p - l for p, l in zip(points, ci_low)],
          [h - p for p, h in zip(points, ci_high)]]
ax.bar(labels, points, color=["steelblue", "darkorange"], edgecolor="white", width=0.5)
ax.errorbar(labels, points, yerr=errors, fmt="none", color="black", capsize=8)
ax.set_title("Nowcast: Before vs After Update", fontsize=12, fontweight="bold")
ax.set_ylabel("GDP Growth (%)")
for i, v in enumerate(points):
    ax.text(i, v + 0.05, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")

# Panel 2: News contributions
update_result.plot_contributions(ax=axes[1])
axes[1].set_title("News Contributions", fontsize=12, fontweight="bold")

# Panel 3: Recession probability comparison
ax = axes[2]
ax.plot(horizons, prob_recession, "b-o", label="Before update", linewidth=2)
ax.plot(horizons, prob_recession_updated, "r-s", label="After update", linewidth=2)
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("P(Recession)")
ax.set_title("Recession Probability: Before vs After", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.02)

fig.suptitle("Real-Time Monitoring: Impact of New Data Release",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print(f"New data release shifted the GDP nowcast by {update_result.total_revision:+.4f}.")
print(f"P(recession at Q+4): {prob_recession[3]:.3f} -> {prob_recession_updated[3]:.3f}")

## Exercise 1 — SOLUTION: MIDAS vs DFM Nowcasting Comparison

Replace DFM with **MIDAS (Mixed Data Sampling)** for the nowcasting step.
Compare accuracy of both nowcasters, then bridge the better one to the VAR
for scenario analysis.

MIDAS uses parameterized weight functions (Beta, Almon) to handle mixed frequencies,
while DFM uses latent factors via the Kalman filter. Each has distinct strengths:
- **DFM**: better when many indicators share a common factor
- **MIDAS**: better when a few specific high-frequency variables drive the target

In [ ]:
# ============================================================
# Exercise 1 — SOLUTION: MIDAS vs DFM Comparison
# ============================================================

from forecastbox.nowcasting import MIDAS

# --- 1. Fit MIDAS model ---
high_freq_vars = ["industrial_production", "retail_sales", "confidence_index"]

midas = MIDAS(
    target="gdp_growth",
    high_freq=high_freq_vars,
    weight_scheme="beta",
    n_lags=12,
    poly_order=2,
    freq_ratio=3,  # quarterly/monthly
)
midas.fit(mixed_freq)
midas_nowcast = midas.nowcast()

print("=== MIDAS Nowcast ===")
print(f"Point estimate: {midas_nowcast.point[0]:.4f}")
if midas_nowcast.lower_80 is not None:
    print(f"80% CI: [{midas_nowcast.lower_80[0]:.4f}, {midas_nowcast.upper_80[0]:.4f}]")
if midas_nowcast.lower_95 is not None:
    print(f"95% CI: [{midas_nowcast.lower_95[0]:.4f}, {midas_nowcast.upper_95[0]:.4f}]")

# Print MIDAS weights
print(f"\nMIDAS weights (Beta scheme):")
print(f"  Weights shape: {midas.weights_.shape}")
print(f"  First 6 weights: {midas.weights_[:6].round(4)}")

# --- 2. DFM Nowcast (already computed above) ---
dfm_nowcast_val = float(nowcast.point[0])
midas_nowcast_val = float(midas_nowcast.point[0])

print(f"\n=== Comparison ===")
print(f"DFM nowcast:   {dfm_nowcast_val:.4f}")
print(f"MIDAS nowcast: {midas_nowcast_val:.4f}")
print(f"Difference:    {midas_nowcast_val - dfm_nowcast_val:+.4f}")

In [ ]:
# --- 3. Pseudo-out-of-sample accuracy comparison ---
# Use expanding window: fit on data up to each quarter, nowcast, compare to actual

gdp_quarterly = mixed_freq["gdp_growth"].dropna()
n_quarters = len(gdp_quarterly)
eval_start = max(20, n_quarters - 8)  # evaluate on last 8 quarters

dfm_errors = []
midas_errors = []

for t in range(eval_start, n_quarters):
    # Get quarterly date for this evaluation point
    target_date = gdp_quarterly.index[t]
    actual_gdp = gdp_quarterly.iloc[t]

    # Create pseudo-real-time dataset: mask GDP at target_date
    rt_data = mixed_freq.loc[:target_date].copy()
    rt_data.loc[target_date, "gdp_growth"] = np.nan

    # DFM nowcast
    dfm_rt = DFMNowcaster(
        n_factors=1, factor_lags=2,
        frequency_map=frequency_map,
        aggregation="sum", em_iterations=100, em_tol=1e-6,
    )
    dfm_rt.fit(rt_data)
    dfm_nc = dfm_rt.nowcast(target="gdp_growth")
    dfm_errors.append(actual_gdp - dfm_nc.point[0])

    # MIDAS nowcast
    midas_rt = MIDAS(
        target="gdp_growth",
        high_freq=high_freq_vars,
        weight_scheme="beta",
        n_lags=12, poly_order=2, freq_ratio=3,
    )
    midas_rt.fit(rt_data)
    midas_nc = midas_rt.nowcast()
    midas_errors.append(actual_gdp - midas_nc.point[0])

dfm_errors = np.array(dfm_errors)
midas_errors = np.array(midas_errors)

print("=== Pseudo-OOS Accuracy Comparison ===")
print(f"Evaluation window: {n_quarters - eval_start} quarters")
print(f"\nDFM:   MAE={np.abs(dfm_errors).mean():.4f}, "
      f"RMSE={np.sqrt((dfm_errors**2).mean()):.4f}, "
      f"Bias={dfm_errors.mean():+.4f}")
print(f"MIDAS: MAE={np.abs(midas_errors).mean():.4f}, "
      f"RMSE={np.sqrt((midas_errors**2).mean()):.4f}, "
      f"Bias={midas_errors.mean():+.4f}")

# Which is better?
dfm_rmse = np.sqrt((dfm_errors**2).mean())
midas_rmse = np.sqrt((midas_errors**2).mean())
best_nowcaster = "DFM" if dfm_rmse <= midas_rmse else "MIDAS"
best_nowcast_val = dfm_nowcast_val if best_nowcaster == "DFM" else midas_nowcast_val
print(f"\nBest nowcaster: {best_nowcaster} (RMSE={min(dfm_rmse, midas_rmse):.4f})")

In [ ]:
# --- 4. Bridge best nowcaster to VAR for scenarios ---

cf_ex1 = ConditionalForecast(var_model, method="analytic")

# Bridge with DFM
fc_dfm_bridge = cf_ex1.forecast(
    steps=forecast_steps,
    conditions={"gdp_growth": [dfm_nowcast_val]},
    n_draws=1000, seed=42,
)

# Bridge with MIDAS
fc_midas_bridge = cf_ex1.forecast(
    steps=forecast_steps,
    conditions={"gdp_growth": [midas_nowcast_val]},
    n_draws=1000, seed=42,
)

# Scenarios using best nowcaster
builder_ex1 = ScenarioBuilder(var_model)
builder_ex1.add_scenario("baseline",
                         {"gdp_growth": [best_nowcast_val]},
                         description=f"Baseline from {best_nowcaster} nowcast")

# Hawkish
hawkish_ff = [last_ff + (150 / 100) * (t + 1) / forecast_steps for t in range(forecast_steps)]
builder_ex1.add_scenario("hawkish",
                         {"gdp_growth": [best_nowcast_val], "fed_funds": hawkish_ff},
                         description="Hawkish: +150bps")

# Dovish
dovish_ff = [last_ff - (75 / 100) * (t + 1) / forecast_steps for t in range(forecast_steps)]
builder_ex1.add_scenario("dovish",
                         {"gdp_growth": [best_nowcast_val], "fed_funds": dovish_ff},
                         description="Dovish: -75bps")

scen_ex1 = builder_ex1.run(steps=forecast_steps, n_draws=1000, seed=42)

print(f"Scenarios built using {best_nowcaster} nowcast = {best_nowcast_val:.4f}")

In [ ]:
# --- 5. Professional Dashboard: DFM vs MIDAS ---

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Panel 1: Nowcast comparison
ax = axes[0, 0]
bars = ax.bar(["DFM", "MIDAS"], [dfm_nowcast_val, midas_nowcast_val],
              color=["steelblue", "darkorange"], edgecolor="white", width=0.5)
ax.axhline(0, color="gray", linestyle="-", linewidth=0.5)
for bar, val in zip(bars, [dfm_nowcast_val, midas_nowcast_val]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
ax.set_title("Current-Quarter GDP Nowcast", fontsize=12, fontweight="bold")
ax.set_ylabel("GDP Growth (%)")
ax.grid(True, alpha=0.3)

# Panel 2: OOS errors comparison
ax = axes[0, 1]
eval_quarters = list(range(1, len(dfm_errors) + 1))
ax.bar(np.array(eval_quarters) - 0.2, np.abs(dfm_errors), 0.35,
       label=f"DFM (RMSE={dfm_rmse:.4f})", color="steelblue", alpha=0.8)
ax.bar(np.array(eval_quarters) + 0.2, np.abs(midas_errors), 0.35,
       label=f"MIDAS (RMSE={midas_rmse:.4f})", color="darkorange", alpha=0.8)
ax.set_xlabel("Evaluation Quarter")
ax.set_ylabel("|Nowcast Error|")
ax.set_title("Pseudo-OOS Absolute Errors", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: MIDAS weight function
ax = axes[0, 2]
lags = np.arange(len(midas.weights_))
ax.bar(lags, midas.weights_, color="darkorange", alpha=0.8, edgecolor="white")
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Weight")
ax.set_title("MIDAS Beta Weight Function", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)

# Panel 4: Bridged forecasts comparison
ax = axes[1, 0]
horizons_plot = np.arange(1, forecast_steps + 1)
ax.plot(horizons_plot, fc_dfm_bridge["gdp_growth"].point, "b-o",
        label=f"DFM bridge (nc={dfm_nowcast_val:.2f})", linewidth=2)
ax.plot(horizons_plot, fc_midas_bridge["gdp_growth"].point, "r-s",
        label=f"MIDAS bridge (nc={midas_nowcast_val:.2f})", linewidth=2)
ax.set_title("GDP Forecast: DFM vs MIDAS Bridge", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 5: Scenarios using best nowcaster
ax = axes[1, 1]
scen_colors = {"baseline": "blue", "hawkish": "red", "dovish": "green"}
for sn in ["baseline", "hawkish", "dovish"]:
    fc = scen_ex1.get(sn, "gdp_growth")
    ax.plot(horizons_plot, fc.point, "-o", color=scen_colors[sn],
            label=sn.capitalize(), linewidth=2, markersize=4)
    if fc.lower_80 is not None:
        ax.fill_between(horizons_plot, fc.lower_80, fc.upper_80,
                        alpha=0.1, color=scen_colors[sn])
ax.set_title(f"GDP Scenarios (from {best_nowcaster})", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 6: Accuracy summary table
ax = axes[1, 2]
ax.axis("off")
tbl_data = [
    ["DFM", f"{np.abs(dfm_errors).mean():.4f}", f"{dfm_rmse:.4f}",
     f"{dfm_errors.mean():+.4f}", f"{dfm_nowcast_val:.4f}"],
    ["MIDAS", f"{np.abs(midas_errors).mean():.4f}", f"{midas_rmse:.4f}",
     f"{midas_errors.mean():+.4f}", f"{midas_nowcast_val:.4f}"],
]
tbl = ax.table(cellText=tbl_data,
               colLabels=["Model", "MAE", "RMSE", "Bias", "Nowcast"],
               cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 2.0)
ax.set_title("Nowcasting Accuracy Summary", fontsize=12, fontweight="bold", pad=20)

fig.suptitle("=== DFM vs MIDAS: Nowcasting Comparison Dashboard ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Reference values
print("\n=== Reference Values: DFM vs MIDAS ===")
print(f"DFM nowcast: {dfm_nowcast_val:.4f} (RMSE={dfm_rmse:.4f})")
print(f"MIDAS nowcast: {midas_nowcast_val:.4f} (RMSE={midas_rmse:.4f})")
print(f"Best nowcaster: {best_nowcaster}")
print(f"Evaluation quarters: {n_quarters - eval_start}")

### Interpretation: Exercise 1

**DFM vs MIDAS Comparison:**

1. **DFM** extracts a latent common factor from all indicators simultaneously using
   the Kalman filter. It naturally handles ragged edges and mixed frequencies. It works
   well when indicators share a strong common cycle (as in GDP tracking).

2. **MIDAS** uses parameterized weight functions (Beta polynomial) to map high-frequency
   lags into a low-frequency regression. It is more parsimonious and interpretable when
   a small number of indicators drive the target variable.

3. **Accuracy**: The pseudo-out-of-sample evaluation reveals which approach works better
   for this specific dataset. The RMSE and bias metrics inform the choice.

4. **Scenario Bridge**: Using the better nowcaster as the bridge to the VAR produces
   more accurate conditional scenarios. The choice of nowcaster matters most for
   short-horizon predictions (Q+1, Q+2).

5. **Practical Recommendation**: In production, run both models and monitor their
   real-time track records. Use the better performer or a simple average as the
   bridge input. Central banks often maintain multiple nowcasting models as cross-checks.

## Exercise 2 — SOLUTION: Conditional Stress Test

Build a **stress test conditional on the nowcast**:

- If nowcast GDP < 0: apply **recession** scenario (GDP -3 sigma, unemployment +2 sigma)
- If nowcast GDP > 2: apply **boom** scenario (GDP +2 sigma, rates +1.5 sigma)
- Otherwise: apply **baseline** (mild shocks)

Compute P(recession deepening) and P(recovery), visualize with conditional fan charts.
This stress test framework is used by bank supervisors (e.g., Fed CCAR/DFAST).

In [ ]:
# ============================================================
# Exercise 2 — SOLUTION: Conditional Stress Test
# ============================================================

from forecastbox.scenarios import StressTest

# --- 1. Determine regime from nowcast ---
nc_val = best_nowcast_val  # use best nowcaster from Exercise 1
print(f"Current GDP nowcast: {nc_val:.4f}")

if nc_val < 0:
    regime = "recession"
    print(f"Regime: RECESSION (nowcast < 0)")
elif nc_val > 2:
    regime = "boom"
    print(f"Regime: BOOM (nowcast > 2)")
else:
    regime = "baseline"
    print(f"Regime: BASELINE (0 <= nowcast <= 2)")

# --- 2. Build stress test ---
stress = StressTest(var_model)

if regime == "recession":
    # Recession deepening: large negative GDP shock + unemployment shock
    stress.add_shock("gdp_growth", magnitude=-3.0, shock_type="std_dev",
                     period=1, duration=2, decay=0.3)
    stress.add_shock("unemployment", magnitude=2.0, shock_type="std_dev",
                     period=1, duration=3, decay=0.2)
    print("Applied recession shocks: GDP -3 sigma (2Q), unemployment +2 sigma (3Q)")
elif regime == "boom":
    # Boom overheating: GDP shock + rate hike
    stress.add_shock("gdp_growth", magnitude=2.0, shock_type="std_dev",
                     period=1, duration=2, decay=0.3)
    stress.add_shock("fed_funds", magnitude=1.5, shock_type="std_dev",
                     period=1, duration=4, decay=0.1)
    print("Applied boom shocks: GDP +2 sigma (2Q), fed_funds +1.5 sigma (4Q)")
else:
    # Baseline mild stress: moderate GDP shock + mild unemployment
    stress.add_shock("gdp_growth", magnitude=-1.5, shock_type="std_dev",
                     period=1, duration=1, decay=0.5)
    stress.add_shock("unemployment", magnitude=1.0, shock_type="std_dev",
                     period=1, duration=2, decay=0.3)
    print("Applied baseline stress: GDP -1.5 sigma (1Q), unemployment +1 sigma (2Q)")

# Run stress test
stress_result = stress.run(steps=forecast_steps, n_draws=1000, seed=42)

# Show impacts
print("\n=== Stress Test Impact Summary ===")
for var in var_vars:
    max_imp, max_h = stress_result.max_impact(var)
    print(f"{var:20s}: max impact = {max_imp:+.4f} at Q+{max_h+1}")

In [ ]:
# --- 3. Monte Carlo: conditional probability assessment ---

mc_stress = MonteCarlo(var_model, n_paths=5000, seed=42, parametric=True)
stress_paths = mc_stress.simulate(steps=forecast_steps)

# Probabilities conditional on current regime
gdp_idx_s = var_vars.index("gdp_growth")
inf_idx_s = var_vars.index("inflation")
unemp_idx_s = var_vars.index("unemployment")

# P(recession deepening) = P(GDP falls further below 0)
# Defined as GDP declining for 2+ consecutive quarters
gdp_paths = stress_paths[:, :, gdp_idx_s]

# Apply the stress shock effect to paths
stressed_gdp = gdp_paths + stress_result.impact["gdp_growth"][np.newaxis, :]
stressed_inf = stress_paths[:, :, inf_idx_s] + stress_result.impact["inflation"][np.newaxis, :]

# P(recession deepening): GDP < 0 for at least 2 consecutive quarters
recession_mask = stressed_gdp < 0
deepening = np.zeros(stressed_gdp.shape[0], dtype=bool)
for t in range(stressed_gdp.shape[1] - 1):
    deepening |= (recession_mask[:, t] & recession_mask[:, t + 1])
p_recession_deepening = deepening.mean()

# P(recovery): GDP > 1% within 4 quarters under stressed conditions
recovery_mask = stressed_gdp[:, :4] > 1.0
p_recovery = recovery_mask.any(axis=1).mean()

# P(recession) by horizon under stressed conditions
p_recession_by_h = (stressed_gdp < 0).mean(axis=0)

# P(high inflation) under stress
p_high_inf_stress = (stressed_inf > 4.0).mean(axis=0)

print("=== Conditional Probabilities Under Stress ===")
print(f"Current regime: {regime.upper()}")
print(f"Nowcast GDP: {nc_val:.4f}")
print(f"\nP(recession deepening — GDP<0 for 2+ quarters): {p_recession_deepening:.4f}")
print(f"P(recovery — GDP>1% within 4Q): {p_recovery:.4f}")
print(f"\nP(recession) by horizon:")
for h_step in range(forecast_steps):
    print(f"  Q+{h_step+1}: {p_recession_by_h[h_step]:.4f}")

In [ ]:
# --- 4. Expected shortfall for inflation ---

# Expected Shortfall (CVaR) at alpha=0.05 (worst 5% of outcomes)
alpha_es = 0.05
es_inflation = np.zeros(forecast_steps)
var_inflation = np.zeros(forecast_steps)  # Value-at-Risk

for t in range(forecast_steps):
    inf_draws = stressed_inf[:, t]
    # VaR: 95th percentile (upper tail risk for inflation)
    var_inflation[t] = np.percentile(inf_draws, 100 * (1 - alpha_es))
    # ES: expected value conditional on exceeding VaR
    tail = inf_draws[inf_draws >= var_inflation[t]]
    es_inflation[t] = tail.mean() if len(tail) > 0 else var_inflation[t]

print("=== Inflation Risk Metrics Under Stress ===")
print(f"{'Horizon':<10} {'VaR(95%)':<12} {'ES(95%)':<12} {'Mean':<12} {'Std':<12}")
print("-" * 58)
for t in range(forecast_steps):
    inf_t = stressed_inf[:, t]
    print(f"Q+{t+1:<8} {var_inflation[t]:<12.4f} {es_inflation[t]:<12.4f} "
          f"{inf_t.mean():<12.4f} {inf_t.std():<12.4f}")

In [ ]:
# --- 5. Conditional Fan Charts and Dashboard ---

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
horizons_st = np.arange(1, forecast_steps + 1)

# Panel 1: Baseline vs Stressed GDP paths
ax = axes[0, 0]
ax.plot(horizons_st, stress_result.baseline["gdp_growth"].point, "b-o",
        label="Baseline", linewidth=2)
ax.plot(horizons_st, stress_result.stressed["gdp_growth"].point, "r-s",
        label="Stressed", linewidth=2)
if stress_result.baseline["gdp_growth"].lower_80 is not None:
    ax.fill_between(horizons_st,
                    stress_result.baseline["gdp_growth"].lower_80,
                    stress_result.baseline["gdp_growth"].upper_80,
                    alpha=0.1, color="blue")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("GDP Growth: Baseline vs Stressed", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Impact by variable
ax = axes[0, 1]
for var in var_vars:
    ax.plot(horizons_st, stress_result.impact[var], "-o", label=var.replace("_", " ").title(),
            linewidth=2, markersize=4)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Impulse Response (Stress Impact)", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Impact (pp)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: Conditional fan chart for GDP
ax = axes[0, 2]
quantiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
fan_data = np.percentile(stressed_gdp, [q * 100 for q in quantiles], axis=0)
# Fan bands
alphas = [0.10, 0.15, 0.25]
for i, (lo_idx, hi_idx) in enumerate([(0, 6), (1, 5), (2, 4)]):
    ax.fill_between(horizons_st, fan_data[lo_idx], fan_data[hi_idx],
                    alpha=alphas[i], color="steelblue")
ax.plot(horizons_st, fan_data[3], "steelblue", linewidth=2, label="Median")
ax.axhline(0, color="red", linestyle="--", linewidth=1.5, label="Recession threshold")
ax.set_title("Conditional Fan Chart: GDP Under Stress", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("GDP Growth (%)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 4: P(recession) and P(recovery)
ax = axes[1, 0]
ax.plot(horizons_st, p_recession_by_h, "r-o", label="P(GDP<0)", linewidth=2)
ax.plot(horizons_st, (stressed_gdp > 1.0).mean(axis=0), "g-s",
        label="P(GDP>1%)", linewidth=2)
ax.axhline(p_recession_deepening, color="red", linestyle=":", alpha=0.5,
           label=f"P(deepening)={p_recession_deepening:.3f}")
ax.axhline(p_recovery, color="green", linestyle=":", alpha=0.5,
           label=f"P(recovery)={p_recovery:.3f}")
ax.set_title("Event Probabilities Under Stress", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Probability")
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 5: Inflation VaR and ES
ax = axes[1, 1]
ax.plot(horizons_st, stressed_inf.mean(axis=0), "b-o", label="Mean", linewidth=2)
ax.plot(horizons_st, var_inflation, "orange", marker="s", label="VaR(95%)", linewidth=2)
ax.plot(horizons_st, es_inflation, "r-^", label="ES(95%)", linewidth=2)
ax.fill_between(horizons_st, var_inflation, es_inflation, alpha=0.2, color="red",
                label="Tail risk zone")
ax.set_title("Inflation: VaR and Expected Shortfall", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (quarters)")
ax.set_ylabel("Inflation (%)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 6: Summary table
ax = axes[1, 2]
ax.axis("off")
summary_data = [
    ["Regime", regime.upper()],
    ["Nowcast GDP", f"{nc_val:.4f}"],
    ["P(Recession Deepening)", f"{p_recession_deepening:.4f}"],
    ["P(Recovery within 4Q)", f"{p_recovery:.4f}"],
    ["Max GDP Impact", f"{stress_result.max_impact('gdp_growth')[0]:+.4f}"],
    ["Max Inflation Impact", f"{stress_result.max_impact('inflation')[0]:+.4f}"],
    ["Inflation ES(95%) at Q+4", f"{es_inflation[3]:.4f}"],
    ["Inflation VaR(95%) at Q+4", f"{var_inflation[3]:.4f}"],
]
tbl_s = ax.table(cellText=summary_data,
                 colLabels=["Metric", "Value"],
                 cellLoc="center", loc="center")
tbl_s.auto_set_font_size(False)
tbl_s.set_fontsize(10)
tbl_s.scale(1.0, 1.8)
ax.set_title("Stress Test Summary", fontsize=12, fontweight="bold", pad=20)

fig.suptitle(f"=== CONDITIONAL STRESS TEST DASHBOARD (Regime: {regime.upper()}) ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Reference values
print("\n=== Reference Values: Conditional Stress Test ===")
print(f"Regime: {regime}")
print(f"Nowcast: {nc_val:.4f}")
print(f"P(recession deepening): {p_recession_deepening:.4f}")
print(f"P(recovery within 4Q): {p_recovery:.4f}")
print(f"GDP max stress impact: {stress_result.max_impact('gdp_growth')[0]:+.4f} at Q+{stress_result.max_impact('gdp_growth')[1]+1}")
print(f"Inflation ES(95%) at Q+4: {es_inflation[3]:.4f}")
print(f"Inflation VaR(95%) at Q+4: {var_inflation[3]:.4f}")

### Interpretation: Exercise 2

**Conditional Stress Test Results:**

1. **Regime Detection**: The nowcast determines the current economic regime. This is
   critical because stress testing should be state-dependent — the same shock has
   different effects depending on whether the economy starts from expansion or contraction.

2. **Shock Design**: Following bank supervisory standards (Fed CCAR/DFAST), we calibrate
   shocks in standard deviations of the VAR residuals. The -3 sigma GDP shock represents
   a severe adverse scenario (roughly a 2008-type event).

3. **Impulse Responses**: The VAR propagates shocks through the system. A GDP shock
   affects inflation (demand channel), unemployment (Okun's law), and interest rates
   (Taylor rule response). The decay parameter controls how quickly the initial shock
   dissipates.

4. **Probability Assessment**:
   - **P(recession deepening)**: probability of GDP staying negative for 2+ consecutive
     quarters under the stress scenario.
   - **P(recovery)**: probability of GDP exceeding 1% within 4 quarters, indicating
     a bounce-back from the shock.

5. **Tail Risk Metrics**:
   - **VaR(95%)**: the inflation level that is exceeded only 5% of the time.
   - **Expected Shortfall (ES)**: the average inflation in the worst 5% of outcomes.
     ES is always >= VaR and captures the severity of extreme scenarios.

6. **Conditional Fan Chart**: The fan chart shows the full distribution of GDP outcomes
   under stress, with the recession threshold at zero. The width of the bands captures
   parameter uncertainty and shock propagation uncertainty.

7. **Policy Use**: This framework helps central banks and regulators assess whether
   financial institutions have sufficient capital buffers to withstand adverse scenarios
   conditional on the current economic assessment (nowcast).